<div style="font-size: 0.85em;">

<h3>Contextual Compression / Reranking</h3>

<p><strong>What is it?</strong><br>
After retrieving chunks, some may be partly irrelevant or contain extra text.<br>
Contextual compression reduces each chunk to only the parts that are relevant to the query.<br>
Reranking reorders chunks by relevance before sending them to the LLM.</p>

<p><strong>Visualisation</strong></p>
<pre>
Retriever → Many chunks (some noisy)
        │
        ▼
[Contextual Compressor] → Smaller, focused excerpts
        │
        ▼
[Reranker (optional)] → Sorted by relevance
        │
        ▼
LLM Generator → Better answer with less irrelevant context
</pre>

<p><strong>Why use it?</strong></p>
<ul>
  <li><strong>Reduces token usage</strong> – smaller prompts, lower cost.</li>
  <li><strong>Improves answer quality</strong> – LLM sees only useful context.</li>
  <li><strong>Reduces hallucination</strong> – fewer distractors.</li>
  <li><strong>Faster and cheaper</strong> – especially with long documents.</li>
</ul>

<p><strong>Example</strong><br>
If a retrieved chunk is a long paragraph about agriculture, and the query is “What is maize smut?”, contextual compression keeps only the sentence(s) about maize smut and drops the rest.</p>

<p><strong>Implementation</strong><br>
We will use LangChain’s <code>ContextualCompressionRetriever</code> and <code>LLMChainExtractor</code> (or <code>LLMChainFilter</code>).</p>

</div>

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

Step 2: Load Relevant Documents

In [2]:
# Load only health and agriculture documents (same as before)
health_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/crop_disease.pdf').load()
agri_html = BSHTMLLoader('../../04_data_ingestion_document_processing/data/agriculture.html', open_encoding='utf-8', bs_kwargs={'features': 'html.parser'}).load()
agri_txt = TextLoader('../../04_data_ingestion_document_processing/data/agriculture.txt', encoding='utf-8').load()

# Combine all documents
all_docs = health_pdf + crop_pdf + agri_html + agri_txt

print(f'Loaded {len(all_docs)} relevant documents.')

Loaded 37 relevant documents.


Step 3: Split and Add Metadata

In [3]:
# Create a text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '.', ' ', '']
)

# Split the loaded documents into chunks
chunks = splitter.split_documents(all_docs)

# Add metadata to each chunk
for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'
    
print(f'Created {len(chunks)} chunks.')

Created 185 chunks.


Step 4: Create Vector Store and Base Retriever

In [4]:
# Create embeddings
embeddings = OpenAIEmbeddings()

# Create vector store from chunks (in-memory)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

# Create base retriever (top 4)
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Vector store and base retriever ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store and base retriever ready.


Step 5: Create Contextual Compression Retriever

In [5]:
# Create an LLM for extraction
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Create a compressor that keeps only relevant parts of each chunk
compressor = LLMChainExtractor.from_llm(llm)

# Wrap the base retriever with the compressor
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print('Contextual compression retriever ready.')

Contextual compression retriever ready.


Step 6: Test Retrieval with and without Compression

In [6]:
# Choose a query
query = 'What are Nigeria communicable and infectious diseases?'

# Without compression
raw_docs = base_retriever.invoke(query)

print('Wthout compression:')
for i, doc in enumerate(raw_docs, start=1):
    print(f'{i}. {doc.page_content}')
    print(f'    Sources: {doc.metadata.get("source", "unknown")}')

print('*' * 100)

# With compression
compressed_docs = compression_retriever.invoke(query)
print('\nWith compression:')
for i, doc in enumerate(compressed_docs, start=1):
    print(f'{i}. {doc.page_content}')
    print(f'    Sources: {doc.metadata.get("source", "unknown")}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Wthout compression:
1. scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Health. The major public health challenges Nigeria faces are infectious diseases, control of vector some diseases,
    Sources: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
2. Introduction Practice Points 
 Nigeria is often referred to as the "Giant of  
Africa", owing to its large population and  
economy, with approximately 182 million   
inhabitants.  
 Communicable and infectious diseases are the 
major health problem in Nigeria.
    Sources: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
3. health workforce; medical products, vaccines and 
technologies; information; financing; and services 
delivery.17 
 
In Nigeria communicable and infectious diseases are 
the major health problem. 3 Nigeria has slowly       
entered the era of ‘disease of the a

**Build a Combined Retriever (Multi‑Query + Compression)**

Step 1: Create a MultiQueryRetriever

In [7]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

Step 2: Wrap the MultiQueryRetriever with Contextual Compression

In [8]:
# Create the compression retriever on top of multi_query_retriever
compressed_multi_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_query_retriever
)

Step 3: Test the Same Query

In [9]:
query = 'What are Nigeria communicable and infectious diseases?'

# Retrieve with combined retriever
docs = compressed_multi_retriever.invoke(query)

print('Retrieved compressed chunks from multi-query:')
for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content}')
    print(f'    Source: {doc.metadata.get("source", "unknown")}')

Retrieved compressed chunks from multi-query:
1. The major public health challenges Nigeria faces are infectious diseases, control of vector some diseases,
    Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
2. HIV/AIDS, tuberculosis, malaria, vaccine preventable disease of childhood, diarrheal, acute respiratory infections
    Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
3.  Communicable and infectious diseases are the major health problem in Nigeria.
    Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
4. The top causes of death in Nigeria are; malaria, lower respiratory infections, HIV/AIDS, diarrheal diseases, meningitis, and tuberculosis. Malaria remains the foremost killer disease in Nigeria.
    Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
5.  Malaria remains the fore